### Window Generation 

Window generation and data preparation for model ingestion. Benign and attack traffic are windowed into two separate sets. Each message is turned into a sliding window over consecutive messages. 

**Windows for Benign** would have window size: 32 and stride: 16 due to the volume of data. We will be ingesting three main files which contain 10M rows each. 

**Windows for Attack** would have window size: 32 and stride: 2 the attack files are smaller and the injection time is shorter, a small stride produces more windows for the model. 

The feature extractor should take a feature array with the shape [num_windows, window_size, num_features] and a target array with the shape [num_windows]. The return value should be the transformed feature and target arrays. 

Structure should follow (num_windows, window_size, features). 

#### Feature Vector 

Every message, benign or attack, is converted into the same 11-feature vector. Keeping one shared representation lets the model learn each attack's behaviour from the same inputs, and matches the features examined in the Attack EDA and Benign EDA. 

11 Features contain: [ CAN_ID , b0, b1, b2, b3, b4, b5, b6, b7, DLC, Δt ] 

- **CAN_ID** — identifier as an integer
- **b0–b7** — payload bytes, padded to 8 bytes
- **DLC** — data length code
- **Δt** — inter-arrival time since the previous message


In [1]:
import sys
from pathlib import Path
from collections import Counter
import pandas as pd
import torch

BASE = Path("..").resolve()
sys.path.append(str(BASE / "src"))
from extract_can_fields import extract_can_fields 
from load_attack_file import load_attack_file, ATTACKS

In [ ]:
TRAIN_RATIO = 0.70

ATTACK_WINDOW_SIZE = 32
ATTACK_STRIDE = 2
ATTACK_FILES = list(ATTACKS.keys())

ATTACK_OUTPUT_DIR = BASE / "Data" / "Attack_Windows_Split"
ATTACK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Attack files:", len(ATTACK_FILES))
print(ATTACK_FILES)
print("Output directory:", ATTACK_OUTPUT_DIR)

Attack files: 9
['Steering_angle_attack', 'Brake_warning_attack', 'Power_steering_attack', 'Min_speedometer_attack_1', 'EMS_replay_attack', 'Steering_angle_replay', 'Fuzzing_random_IDs', 'Fuzzing_valid_IDs', 'DoS_attack']
Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/Data/Attack_Windows_Split


### Attack Windows 

Attack windows use the same 11 feature vector as benign traffic [ CAN_ID , b0, b1, b2, b3, b4, b5, b6, b7, DLC, Δt ]. 

Each attack recording stays separate: windows are created per source file, never across files. Mixing files would break the time order of the bus.

Message labels come from load_attack_file: rows with flag=1 get the attack class; rows with flag=0 stay Benign. The injection interval is only used for timing analysis in the EDA, not for window labels.

A window is kept as Attack if it contains **at least one** message with flag=1. Windows with all flag=0 are discarded. The window label is the attack class of that file (DoS / Spoofing / Fuzzing / Replay).

Saved tensors go to the Attack_windows/ folder, with the corresponding .pt window files saved.

In [5]:
rows = []

for name in ATTACK_FILES: 
    df = load_attack_file(name)
    flags = df["flag"].tolist()
    
    total_windows = 0  # Total number of windows
    mixed_windows = 0  # Some flags are 0, some are 1
    pure_attack_windows = 0   # All flags are 1 
    pure_benign_windows = 0  # All flags are 0
    
    # Iterate over the message labels in steps of ATTACK_STRIDE
    for start in range(0, len(flags) - ATTACK_WINDOW_SIZE + 1, ATTACK_STRIDE):
        window_flags = flags[start : start + ATTACK_WINDOW_SIZE]
        # Count the total number of windows
        total_windows += 1 
        
        injected_count = sum(window_flags)
        
        # Check if the window contains both attack and benign messages
        if injected_count == 0: 
            pure_benign_windows += 1 
        elif injected_count == ATTACK_WINDOW_SIZE:
            pure_attack_windows += 1 
        else: 
            mixed_windows += 1 
        
    total_attack_windows = pure_attack_windows + mixed_windows      
    rows.append({
        "file": name, 
        "total_windows": total_windows, 
        "mixed_windows": mixed_windows, 
        "pure_attack_windows": pure_attack_windows, 
        "pure_benign_windows": pure_benign_windows, 
        "total_attack_windows": total_attack_windows,
        "percentage_mixed": round(100 * total_attack_windows / total_windows, 2),
    })

In [6]:
rows_df = pd.DataFrame(rows)
rows_df.head(10)

,file,total_windows,mixed_windows,pure_attack_windows,pure_benign_windows,total_attack_windows,percentage_mixed
0,Steering_angle_attack,184800,63110,0,121690,63110,34.15
1,Brake_warning_attack,294281,109950,0,184331,109950,37.36
2,Power_steering_attack,180584,49712,0,130872,49712,27.53
3,Min_speedometer_attack_1,272787,120336,0,152451,120336,44.11
4,EMS_replay_attack,162058,64393,0,97665,64393,39.73
5,Steering_angle_replay,237889,101481,0,136408,101481,42.66
6,Fuzzing_random_IDs,364488,120733,0,243755,120733,33.12
7,Fuzzing_valid_IDs,128691,22907,0,105784,22907,17.80
8,DoS_attack,170878,88094,0,82784,88094,51.55


- total windows: all attack windows 
- mixed windows: some flag=1 and some flag=0 (still labelled Attack)
- pure attack: windows that contain exclusively attack messages (all flags are 1), this column shows zeros since the attack recordings follow benign background traffic while the malicious messages are being injected into the CAN stream, so there is no sequence with 32 messages which contain only messages with 1. 

- pure benign: windows where all messages are benign (all flags are 0), which can be dropped from the attack dataset.
- percentage mixed: the total percentage of useful attack windows (combining pure attack and mixed windows) kept for training. 

### Flag Based Window Label

Attack windows are labelled 0–3 (DoS, Spoofing, Fuzzing, Replay). Windows with no flag=1 are dropped. 

In [7]:

CLASS_TO_ID = {
    "DoS": 0,
    "Spoofing": 1,
    "Fuzzing": 2,
    "Replay": 3,
}

print("Output directory:", ATTACK_OUTPUT_DIR)
print("Class mapping:", CLASS_TO_ID)


Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/Data/Attack_Windows_Split
Class mapping: {'DoS': 0, 'Spoofing': 1, 'Fuzzing': 2, 'Replay': 3}


#### Processing Files 

In [8]:
def make_attack_windows(df, name, window_size, stride):    
    features = []
    for i in range(len(df)):
        # Convert the payload to a list of integers
        payload_bytes = [int(b, 16) for b in df["payload"].iloc[i]]
        # Create a feature vector for the current message
        features.append([df["id_int"].iloc[i]] + payload_bytes + [df["dlc"].iloc[i], df["dt"].iloc[i]])

    flags = df["flag"].tolist()
    class_id = CLASS_TO_ID[ATTACKS[name]]

    windows, window_labels = [], []
    
    # Iterate over the features in steps of the stride
    for start in range(0, len(features) - window_size + 1, stride):
        # If the window contains no attack messages, skip it
        if sum(flags[start:start + window_size]) < 1:
            continue 
        # Append the window and its label to the lists
        windows.append(features[start:start + window_size])
        window_labels.append(class_id)
        
    return (torch.tensor(windows, dtype=torch.float32), torch.tensor(window_labels, dtype=torch.long),
)

In [9]:
for name in ATTACK_FILES:
    print("Processing file:", name)
    
    df = load_attack_file(name)
    split_index = int(len(df) * TRAIN_RATIO)
    
    train_windows, train_labels = make_attack_windows(df.iloc[:split_index], name, ATTACK_WINDOW_SIZE, ATTACK_STRIDE)
    test_windows, test_labels = make_attack_windows(df.iloc[split_index:], name, ATTACK_WINDOW_SIZE, ATTACK_STRIDE)

    torch.save({"features": train_windows, "labels": train_labels}, ATTACK_OUTPUT_DIR / f"{name}_train.pt",)
    torch.save({"features": test_windows, "labels": test_labels},ATTACK_OUTPUT_DIR / f"{name}_test.pt",)
    
    print("train:", train_windows.shape, "| test:", test_windows.shape)
print("All attack windows have been saved.")

Processing file: Steering_angle_attack
train: torch.Size([61392, 32, 11]) | test: torch.Size([1703, 32, 11])
Processing file: Brake_warning_attack
train: torch.Size([90090, 32, 11]) | test: torch.Size([19860, 32, 11])
Processing file: Power_steering_attack
train: torch.Size([33717, 32, 11]) | test: torch.Size([15983, 32, 11])
Processing file: Min_speedometer_attack_1
train: torch.Size([98687, 32, 11]) | test: torch.Size([21638, 32, 11])
Processing file: EMS_replay_attack
train: torch.Size([44378, 32, 11]) | test: torch.Size([20000, 32, 11])
Processing file: Steering_angle_replay
train: torch.Size([99585, 32, 11]) | test: torch.Size([1881, 32, 11])
Processing file: Fuzzing_random_IDs
train: torch.Size([94621, 32, 11]) | test: torch.Size([26097, 32, 11])
Processing file: Fuzzing_valid_IDs
train: torch.Size([0]) | test: torch.Size([22906, 32, 11])
Processing file: DoS_attack
train: torch.Size([49558, 32, 11]) | test: torch.Size([38521, 32, 11])
All attack windows have been saved.


In [10]:
pt_files = sorted(ATTACK_OUTPUT_DIR.glob("*.pt"))
print("Files found:", len(pt_files))

rows = []
for pt_path in pt_files:
    data = torch.load(pt_path, map_location="cpu")

    features = data["features"]
    labels = data["labels"]

    rows.append({
        "file": pt_path.name,
        "keys": list(data.keys()),
        "features_shape": tuple(features.shape),
        "labels_shape": tuple(labels.shape),
        "dtype": str(features.dtype),
        "label_values": labels.unique().tolist(),
    })
    
pd.DataFrame(rows)

Files found: 18


,file,keys,features_shape,labels_shape,dtype,label_values
0,Brake_warning_attack_test.pt,"[features, labels]","(19860, 32, 11)","(19860,)",torch.float32,[1]
1,Brake_warning_attack_train.pt,"[features, labels]","(90090, 32, 11)","(90090,)",torch.float32,[1]
2,DoS_attack_test.pt,"[features, labels]","(38521, 32, 11)","(38521,)",torch.float32,[0]
3,DoS_attack_train.pt,"[features, labels]","(49558, 32, 11)","(49558,)",torch.float32,[0]
4,EMS_replay_attack_test.pt,"[features, labels]","(20000, 32, 11)","(20000,)",torch.float32,[3]
5,EMS_replay_attack_train.pt,"[features, labels]","(44378, 32, 11)","(44378,)",torch.float32,[3]
6,Fuzzing_random_IDs_test.pt,"[features, labels]","(26097, 32, 11)","(26097,)",torch.float32,[2]
7,Fuzzing_random_IDs_train.pt,"[features, labels]","(94621, 32, 11)","(94621,)",torch.float32,[2]
8,Fuzzing_valid_IDs_test.pt,"[features, labels]","(22906, 32, 11)","(22906,)",torch.float32,[2]
9,Fuzzing_valid_IDs_train.pt,"[features, labels]","(0,)","(0,)",torch.float32,[]
